### Bibliotecas

*   A biblioteca [`requests`](https://pypi.org/project/requests/) do Python é usada para fazer requisições HTTP, permitindo conectar a servidores web, enviar dados e receber respostas;
*   A biblioteca [`pytz`](https://pypi.org/project/pytz/) do Python é usada para trabalhar com fusos horários, permitindo manipulação de datas e horas em diferentes regiões do mundo
*   A biblioteca [`BeautifulSoup`](https://pypi.org/project/beautifulsoup4) do Python é usada para extrair dados de documentos HTML/XML, permitindo navegar, pesquisar e modificar o conteúdo do documento, sendo amplamente utilizada para web scraping.

In [ ]:
import random
import os
import re
import requests
import pytz
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from enum import Enum
from bs4 import BeautifulSoup
from datetime import datetime

### Funções utilitárias

Essas funções serão utilizadas em diferentes partes do notebook.

In [ ]:
agents = ["Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/86.0.4240.198 Safari/537.36", "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:109.0) Gecko/20100101 Firefox/116.0", "Mozilla/5.0 (Linux; Android 10; K) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36", "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/15.6 Safari/605.1.15", "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Edg/115.0.1901.188", "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36 Edg/117.0.0.0", "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36"]

def get_agent():
    number_of_agents = len(agents) - 1
    agent_index = random.randint(0, number_of_agents)
    return agents[agent_index]

class Sources(str, Enum):
    B3 = 'b3'
    CEPEA = 'cepea'
    COOPAMA = 'coopama'
    COOPERCAM = 'coopercam'
    CCMG = 'ccmg'
    COOXUPE = 'cooxupe'
    CAFEPOCOS = 'cafe-pocos'
    COCAPEC = 'cocapec'
    AIBA = 'aiba'
    CCCV = 'cccv'

    def __str__(self) -> str:
        return self.value

def date_time(separator_date='/', separator_time=':'):
   snap = datetime.now(pytz.timezone('America/Sao_Paulo'))

   date = snap.strftime(f"%d{separator_date}%m{separator_date}%Y")
   time = snap.strftime(f"%H{separator_time}%M{separator_time}%S")

   date_time = {
      'date': date,
      'time': time
   }

   return date_time

def retain_numbers_in_string(str_value):
  result = ""
  digits = ",0123456789"
  for i in str_value:
      if i in digits:
          result += i

  return result

def comma_to_point(str_value):
  srt_as_list = list(str_value)

  for index, character in enumerate(srt_as_list):
      if (character == ','):
          srt_as_list[index] = '.'

  return ''.join(srt_as_list)

def br_money_to_float(str_value):
  match = re.search(r'\d{1,3}(?:\.\d{3})*(?:,\d+)?|\d+(?:,\d+)?', str_value)
  if not match:
    return None
  number = match.group(0).replace('.', '').replace(',', '.')
  return round(float(number), digits_for_floats)

def price_noticias_agricolas(headers, url, row_key):
  site = requests.get(url, headers=headers, timeout=30)
  soup = BeautifulSoup(site.content, 'html.parser')

  table = soup.find('table')
  if table is None:
    print(f"Nenhuma tabela encontrada em {url}")
    return None

  for row in table.find_all('tr'):
    cols = row.find_all('td')
    if len(cols) >= 2 and row_key.lower() in cols[0].text.lower():
      return br_money_to_float(cols[1].text)

  print(f"Linha '{row_key}' não encontrada em {url}")
  return None

### Variáveis globais

Essas variáveis serão utilizadas em diferentes partes do notebook.

In [ ]:
headers = {'User-Agent': get_agent()}
digits_for_floats = 2

### Cotações

Buscaremos cotações de preço de café em diferentes regioões/cidades do Brasil.

A idéia é consolidar essas cotações ao longo do tempo para fazer análises.

#### Cotação na B3

A Brasil, Bolsa, Balcão ([B3](https://www.b3.com.br)) é a bolsa de valores brasileira, que fica sediada na cidade de São Paulo.

In [ ]:
def price_b3_via_coox(headers, url):
  us_price = 0
  dollar_price = 0

  try:
    site = requests.get(url, headers=headers)
    soup_content = BeautifulSoup(site.content, 'html.parser')

    rows = soup_content.find_all('tr')
    if not rows:
        print("Nenhuma linha de tabela (tr) encontrada na página.")
        return 0

    def parse_numeric_value(raw_val):
      cleaned = ''.join(c for c in raw_val if c.isdigit() or c in '.,-')
      if ',' in cleaned:
        cleaned = cleaned.replace('.', '').replace(',', '.')
      return float(cleaned)

    for row in rows:
      columns = row.find_all(['td', 'th'])
      if len(columns) > 1:
        desc = columns[0].text.strip()

        if 'DOL COM' in desc:
          dollar_price = parse_numeric_value(columns[1].text.strip())
          dollar_price = round(dollar_price, 4)
        if 'ICF' in desc:
          us_price = parse_numeric_value(columns[1].text.strip())
          us_price = round(us_price, digits_for_floats)

    print(f"Dólar Comercial: {dollar_price}, ICF: {us_price}")
    return round(us_price * dollar_price, digits_for_floats)
  except Exception as error:
    display(error)
    return 0

#### Cotação do CPEA

O [CEPEA](https://www.cepea.esalq.usp.br/br) é o Centro de Estudos Avançados em Economia Aplicada da Escola Superior de Agricultura (Esalq/USP).

In [ ]:
def price_cepea(headers, url):
  price = 0

  try:
    site = requests.get(url, headers=headers)
    soup = BeautifulSoup(site.content, 'html.parser')

    table = soup.find('table', {'id':'imagenet-indicador1'})
    tds = soup.find_all('td')
    price = tds[1].text.strip()
    price = retain_numbers_in_string(price)
    price = comma_to_point(price)
    price = float(price)

    return round(price, digits_for_floats)

  except Exception as error:
    display(error)
    return 0


#### Cotação da Coopama

A [Coopama](https://www.coopama.com.br/) é a Cooperativa Agrária de Machado que, entre outras atividades, oferece comercialização e armazenagem de café e outros grãos, como milho, soja, sorgo e aveia.

In [ ]:
def price_coopama(headers, url):
  price = 0

  try:
    site = requests.get(url, headers=headers)
    soup = BeautifulSoup(site.content, 'html.parser')

    div_cota = soup.find('div', {'id', 'cotacao-cafe' })
    h5s = soup.find_all('h5', {'class': 'linhaBoxCotacao'})
    spans = soup.find_all('span', {'class': 'col'})

    found = False
    for span in spans:
      if (found):
        price = span.text
        price = retain_numbers_in_string(price)
        price = price[:4]
        price = float(price)
        break

      if 'Duro tipo 6' in span.text:
        found = True

    return round(price, digits_for_floats)

  except Exception as error:
    display(error)
    return 0


#### Cotação da Coopercam

A [Coopercam](https://coopercam.com.br/) é a Cooperativa do Cafeicultores de Campos Gerais MG e Campo do Meio MG. Sediado no segundo maior município produtor de café do Brasil, a cooperativa atua em aproximadamente 15 mil ha de café, que abrange os munícipios de Campos Gerais, Campo do Meio, distrito de Córrego do Ouro e cidades circunvizinhas.

In [ ]:
def price_coopercam(headers, url):
  try:
    site = requests.get(url, headers=headers, timeout=30)
    soup = BeautifulSoup(site.content, 'html.parser')

    for link in soup.find_all('a'):
      text = link.get_text(' ', strip=True)
      if text.lower().startswith('duro limpo'):
        return br_money_to_float(text.split(':', 1)[1])

    print("Cotação 'Duro Limpo' não encontrada na Coopercam.")
    return None

  except Exception as error:
    display(error)
    return None

#### Cotação do CCMG

O [CCMG](https://cccmg.com.br/) é o Centro do Comércio de Café do Estado de Minas Gerais, uma instituição representativa, sem fins lucrativos, que agrega empresas que atuam no agronegócio café.

> **Observação:** o site do CCCMG está protegido pelo Cloudflare ("Performing security verification"), que bloqueia requisições automatizadas feitas com `requests`. Por isso a função abaixo tenta a leitura e, se for bloqueada, devolve `None` e a coluna `ccmg` fica vazia no CSV naquele dia (não contornamos a proteção anti-robô do site).

In [ ]:
def price_ccmg(headers, url):
  try:
    site = requests.get(url, headers=headers, timeout=30)

    if site.status_code != 200 or 'Just a moment' in site.text:
      print(f"CCMG bloqueou a requisição (HTTP {site.status_code} - proteção Cloudflare).")
      return None

    soup = BeautifulSoup(site.content, 'html.parser')
    text = soup.get_text(' ', strip=True)

    match = re.search(r'dur[ao][^R]{0,80}R\$\s*([\d\.]+,\d{2})', text, re.IGNORECASE)
    if match:
      return br_money_to_float(match.group(1))

    print("Cotação de bebida dura não encontrada no CCMG.")
    return None

  except Exception as error:
    display(error)
    return None

#### Cotação da Cooxupé

A [Cooxupé](https://www.cooxupe.com.br/) é a Cooperativa Regional de Cafeicultores em Guaxupé MG. Atualmente, é a maior cooperativa de cafeicultores do mundo, contando com mais de 20 mil cooperados espalhados em mais de 330 municípios localizados no Sul de Minas, Cerrado Mineiro, Matas de Minas e Vale do Rio Pardo (no estado de São Paulo).

Ler cotações no [Hub do Café](https://hubdocafe.cooxupe.com.br/cafe-da-tarde/).

In [ ]:
def price_cooxupe(headers, url, fallback_url=None):
  try:
    site = requests.get(url, headers=headers, timeout=30)
    soup = BeautifulSoup(site.content, 'html.parser')

    label = soup.find(string=re.compile(r'Caf[ée] fino Cooxup[ée]', re.IGNORECASE))
    if label:
      value = label.find_next(string=re.compile(r'R\$'))
      if value:
        return br_money_to_float(value)
    if fallback_url:
      print("Hub do Café sem o valor no HTML; usando o Notícias Agrícolas.")
      return price_noticias_agricolas(headers, fallback_url, 'Cooxupé')

    return None

  except Exception as error:
    display(error)
    return None

#### Cotação da Café Poços

A [Café Poços](https://www.cafepocos.com.br/) é a Cooperativa Regional dos Cafeicultores de Poços de Caldas.

Ler cotações no [notícias agrícolas](https://www.noticiasagricolas.com.br/cotacoes/cafe/cafe-arabica-mercado-fisico-tipo-6-duro).

Usamos a tabela **Tipo 6 Bebida Dura Bica Corrida** (mesmo padrão das demais fontes). A página mostra várias tabelas, uma por dia de fechamento; a primeira é sempre a mais recente.

In [ ]:
def price_cafe_pocos(headers, url):
  try:
    return price_noticias_agricolas(headers, url, 'Poços de Caldas')
  except Exception as error:
    display(error)
    return None

### Novas fontes (outros estados produtores)

Além das fontes de Minas Gerais e da B3/CEPEA, adicionamos três fontes de **outros estados**:

| Fonte | Estado | Onde lemos | Produto |
|---|---|---|---|
| Cocapec | São Paulo (Franca / Alta Mogiana) | site da cooperativa | Café Fino |
| AIBA | Bahia (Oeste baiano / Cerrado) | Notícias Agrícolas | Tipo 6 bebida dura |
| CCCV | Espírito Santo (Vitória) | site do CCCV | Arábica bebida dura |

#### Cotação da Cocapec

A [Cocapec](https://www.cocapec.com.br/) é a Cooperativa de Cafeicultores e Agropecuaristas, sediada em **Franca/SP**, na região da Alta Mogiana.

Na página inicial, o carrossel "Cotações" mostra cartões com um título (`h3`), por exemplo **Café Fino**, seguido do preço (`R$ 1.620,00`). Se o cartão não for encontrado, usamos como plano B a linha "Franca/SP (Cocapec)" do Notícias Agrícolas (tabela Tipo 6/7, que traz o mesmo valor).

In [ ]:
def price_cocapec(headers, url, fallback_url=None):
  try:
    site = requests.get(url, headers=headers, timeout=30)
    soup = BeautifulSoup(site.content, 'html.parser')

    for h3 in soup.find_all('h3'):
      if h3.get_text(strip=True).lower() == 'café fino':
        value = h3.find_next(string=re.compile(r'R\$'))
        if value:
          return br_money_to_float(value)

    if fallback_url:
      print("Cartão 'Café Fino' não encontrado no site da Cocapec; usando o Notícias Agrícolas.")
      return price_noticias_agricolas(headers, fallback_url, 'Cocapec')

    return None

  except Exception as error:
    display(error)
    return None

#### Cotação da AIBA (Oeste da Bahia)

A [AIBA](https://aiba.org.br/) é a Associação de Agricultores e Irrigantes da Bahia, que representa os produtores do **Oeste baiano** (café irrigado do Cerrado da Bahia).

O preço é publicado diariamente pelo [Notícias Agrícolas](https://www.noticiasagricolas.com.br/cotacoes/cafe/cafe-arabica-mercado-fisico-tipo-6-duro), na linha "Oeste da Bahia (AIBA)" da tabela Tipo 6 Bebida Dura.

In [ ]:
def price_aiba(headers, url):
  try:
    return price_noticias_agricolas(headers, url, 'AIBA')
  except Exception as error:
    display(error)
    return None

#### Cotação do CCCV (Espírito Santo)

O [CCCV](https://www.cccv.org.br/) é o Centro do Comércio de Café de Vitória, no **Espírito Santo**, produtor de café arábica nas montanhas capixabas.

A página inicial mostra o bloco "Cotação" com parágrafos (`p`): a descrição do café e, logo depois, o preço. Lemos o arábica **bebida "dura"**, bica corrida.

In [ ]:
def price_cccv(headers, url):
  try:
    site = requests.get(url, headers=headers, timeout=30)
    soup = BeautifulSoup(site.content, 'html.parser')

    paragraphs = soup.find_all('p')
    for index, p in enumerate(paragraphs):
      text = p.get_text(' ', strip=True).lower()
      if 'bebida' in text and 'dura' in text:
        for next_p in paragraphs[index + 1:index + 3]:
          if 'R$' in next_p.text:
            return br_money_to_float(next_p.text)

    print("Cotação do arábica bebida dura não encontrada no CCCV.")
    return None

  except Exception as error:
    display(error)
    return None

### Leitura dos Dados

In [ ]:
b3 = True
cepea = True
coopama = True
coopercam = True
ccmg = True
cooxupe = True
cafe_pocos = True
cocapec = True
aiba = True
cccv = True

URL_NA_TIPO6_DURO = 'https://www.noticiasagricolas.com.br/cotacoes/cafe/cafe-arabica-mercado-fisico-tipo-6-duro'
URL_NA_TIPO6_7 = 'https://www.noticiasagricolas.com.br/cotacoes/cafe/cafe-arabica-mercado-fisico-tipo-6-7'
URL_NA_CEREJA = 'https://www.noticiasagricolas.com.br/cotacoes/cafe/cafe-cereja-descascado-mercado-fisico'

prices =  {
    'data': [],
    Sources.B3: [],
    Sources.COOPAMA: [],
    Sources.CEPEA: [],
    Sources.COOPERCAM: [],
    Sources.CCMG: [],
    Sources.COOXUPE: [],
    Sources.CAFEPOCOS: [],
    Sources.COCAPEC: [],
    Sources.AIBA: [],
    Sources.CCCV: []
}

def valid(price):
  return price if price else None

date = date_time()['date']

prices['data'].append(date)

if (b3):
  b3_price = price_b3_via_coox(headers, 'https://hubdocafe.cooxupe.com.br')
  prices[Sources.B3].append(valid(b3_price))

if(coopama):
  coop_price = price_coopama(headers, 'https://www.coopama.com.br/')
  prices[Sources.COOPAMA].append(valid(coop_price))

if (cepea):
  cepea_price = price_cepea(headers, 'https://www.cepea.org.br/br/indicador/cafe.aspx')
  prices[Sources.CEPEA].append(valid(cepea_price))

if (coopercam):
  coopercam_price = price_coopercam(headers, 'https://coopercam.com.br/cotacoes/')
  prices[Sources.COOPERCAM].append(valid(coopercam_price))

if (ccmg):
  ccmg_price = price_ccmg(headers, 'https://cccmg.com.br/')
  prices[Sources.CCMG].append(valid(ccmg_price))

if (cooxupe):
  cooxupe_price = price_cooxupe(headers, 'https://hubdocafe.cooxupe.com.br/cafe-da-tarde/', URL_NA_CEREJA)
  prices[Sources.COOXUPE].append(valid(cooxupe_price))

if (cafe_pocos):
  cafe_pocos_price = price_cafe_pocos(headers, URL_NA_TIPO6_DURO)
  prices[Sources.CAFEPOCOS].append(valid(cafe_pocos_price))

if (cocapec):
  cocapec_price = price_cocapec(headers, 'https://www.cocapec.com.br/', URL_NA_TIPO6_7)
  prices[Sources.COCAPEC].append(valid(cocapec_price))

if (aiba):
  aiba_price = price_aiba(headers, URL_NA_TIPO6_DURO)
  prices[Sources.AIBA].append(valid(aiba_price))

if (cccv):
  cccv_price = price_cccv(headers, 'https://www.cccv.org.br/')
  prices[Sources.CCCV].append(valid(cccv_price))

for key, values in prices.items():
  if not values:
    values.append(None)

display(prices)

### Salvamento dos Dados

In [ ]:
import os
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

folder_path = '/content/drive/MyDrive/Anos/2026/2o Semestre/Ciência de Dados/precos_cafe'
file_name = os.path.join(folder_path, 'precos_cafe.csv')

os.makedirs(folder_path, exist_ok=True)

df_new = pd.DataFrame({str(key): value for key, value in prices.items()})

if os.path.exists(file_name):
    df_existing = pd.read_csv(file_name)
    df_final = pd.concat([df_existing, df_new], ignore_index=True)
    df_final = df_final.drop_duplicates(subset='data', keep='last')
    df_final.to_csv(file_name, index=False)
    print(f"Dados adicionados em: {file_name}")
else:
    df_new.to_csv(file_name, index=False)
    print(f"Arquivo criado em: {file_name}")

### Visualização dos Dados

In [ ]:
df_saved = pd.read_csv(file_name)

df_saved['data_dt'] = pd.to_datetime(df_saved['data'], format='%d/%m/%Y')
df_saved = df_saved.sort_values('data_dt').drop(columns='data_dt').reset_index(drop=True)
display(df_saved)

df_long_form = df_saved.melt('data', var_name='fonte', value_name='preco')
display(df_long_form)

#### Gráfico de Evolução de Preços

Gráfico de Linhas.

In [ ]:
sns.set(font_scale=1)
plt.figure(figsize=(15, 6))

graph = sns.lineplot(
    data=df_long_form,
    x='data',
    y='preco',
    hue='fonte',
    style='fonte',
    markers=True,
    dashes=False,
    palette='tab10'
)

graph.set_ylim(1400,2000)
graph.set(title='Preços do Café no Brasil')
graph.set(ylabel='Preço (R$/60Kg)')
graph.set(xlabel='Dia')
graph.grid(True, linestyle='--', alpha=0.5)

legends, labels = graph.get_legend_handles_labels()
graph.legend(legends, [label.upper() for label in labels], title="Entidade", bbox_to_anchor=(1.01, 1), loc='upper left')

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

#### Gráfico de Distribuição de Preços por Fonte

Gráfico de caixas (boxplot) com os pontos de cada dia por cima.

Ele mostra, para cada fonte:
* o **nível de preço** (mediana, linha central da caixa);
* a **variação no período** (altura da caixa e dos "bigodes");
* a **região/estado** de cada fonte (cor).

Com ele dá para ver, por exemplo, que a B3 (bolsa) fica acima de todas as praças físicas, que o Oeste da Bahia (AIBA) e o Espírito Santo (CCCV) costumam pagar menos que o Sul de Minas, e quais fontes oscilam mais.

In [ ]:
estados = {
    'b3': 'Bolsa (B3)',
    'cepea': 'SP (indicador)',
    'coopama': 'MG',
    'coopercam': 'MG',
    'ccmg': 'MG',
    'cooxupe': 'MG',
    'cafe-pocos': 'MG',
    'cocapec': 'SP',
    'aiba': 'BA',
    'cccv': 'ES'
}

df_box = df_long_form.dropna(subset=['preco']).copy()
df_box['regiao'] = df_box['fonte'].map(estados).fillna('Outros')

ordem = df_box.groupby('fonte')['preco'].median().sort_values(ascending=False).index

sns.set(font_scale=1)
plt.figure(figsize=(15, 6))

box = sns.boxplot(
    data=df_box,
    x='fonte',
    y='preco',
    hue='regiao',
    order=ordem,
    dodge=False,
    palette='Set2',
    showfliers=False
)
sns.stripplot(
    data=df_box,
    x='fonte',
    y='preco',
    order=ordem,
    color='black',
    size=5,
    alpha=0.6,
    jitter=0.15
)

box.set(title='Distribuição dos Preços do Café por Fonte')
box.set(ylabel='Preço (R$/60Kg)')
box.set(xlabel='Fonte')
box.set_xticks(range(len(ordem)))
box.set_xticklabels([f.upper() for f in ordem])
box.grid(True, axis='y', linestyle='--', alpha=0.5)
box.legend(title='Estado / Região', bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout()
plt.show()

resumo = df_box.groupby('fonte')['preco'].agg(['count', 'mean', 'min', 'max', 'std']).round(2)
resumo = resumo.rename(columns={'count': 'dias', 'mean': 'média', 'min': 'mínimo', 'max': 'máximo', 'std': 'desvio padrão'})
display(resumo.loc[ordem])